# Qwen2.5-Coder-14B-Instruct Inference API Server
Runs a FastAPI server on Kaggle and exposes it via ngrok.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q fastapi uvicorn pyngrok transformers accelerate torch

In [ ]:
# ── 2. Authenticate ngrok ────────────────────────────────────────────────────
# Get your free token at https://dashboard.ngrok.com/get-started/your-authtoken
# Option A: paste token directly.
# Option B: store as a Kaggle Secret named NGROK_TOKEN and uncomment below.
#   from kaggle_secrets import UserSecretsClient
#   NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_TOKEN")
import os
from pyngrok import ngrok, conf

NGROK_TOKEN = os.environ.get("NGROK_TOKEN", "PASTE_YOUR_TOKEN_HERE")
conf.get_default().auth_token = NGROK_TOKEN

In [ ]:
# ── 3. Load Qwen2.5-Coder-14B-Instruct ──────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (this may take a few minutes on first run)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

In [ ]:
# ── 4. Define FastAPI app ────────────────────────────────────────────────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Literal, Optional

app = FastAPI(title="Qwen2.5-Coder-14B-Instruct Inference API", version="1.0")


# ── Shared generation params ─────────────────────────────────────────────────
class GenParams(BaseModel):
    max_new_tokens: Optional[int] = Field(default=512, ge=1, le=4096)
    temperature: Optional[float] = Field(default=0.7, ge=0.0, le=2.0)
    top_p: Optional[float] = Field(default=0.95, ge=0.0, le=1.0)
    do_sample: Optional[bool] = True


# ── /chat  (instruct / chat-template endpoint) ────────────────────────────────
class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str

class ChatRequest(GenParams):
    messages: List[Message]

class ChatResponse(BaseModel):
    role: str
    content: str
    num_new_tokens: int


@app.post("/chat", response_model=ChatResponse)
def chat(req: ChatRequest):
    try:
        text = tokenizer.apply_chat_template(
            [m.model_dump() for m in req.messages],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature if req.do_sample else 1.0,
                top_p=req.top_p if req.do_sample else 1.0,
                do_sample=req.do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_len:]
        reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
        return ChatResponse(role="assistant", content=reply, num_new_tokens=len(new_tokens))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── /generate  (raw completion, no chat template) ────────────────────────────
class GenerateRequest(GenParams):
    prompt: str

class GenerateResponse(BaseModel):
    prompt: str
    generated_text: str
    num_new_tokens: int


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    try:
        inputs = tokenizer(req.prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature if req.do_sample else 1.0,
                top_p=req.top_p if req.do_sample else 1.0,
                do_sample=req.do_sample,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_len:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        return GenerateResponse(prompt=req.prompt, generated_text=generated_text, num_new_tokens=len(new_tokens))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

In [ ]:
# ── 5. Start uvicorn in a background thread ──────────────────────────────────
import threading
import uvicorn

PORT = 8000

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print(f"Uvicorn started on port {PORT}")

In [ ]:
# ── 6. Expose with ngrok ─────────────────────────────────────────────────────
import time
time.sleep(2)  # give uvicorn a moment to bind

tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("=" * 60)
print(f"  Public URL : {public_url}")
print(f"  Health     : {public_url}/health")
print(f"  Docs       : {public_url}/docs")
print(f"  Chat       : POST {public_url}/chat")
print(f"  Generate   : POST {public_url}/generate")
print("=" * 60)

In [ ]:
# ── 7. Quick smoke-test (/chat endpoint) ─────────────────────────────────────
import requests, json

resp = requests.post(
    f"{public_url}/chat",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user",   "content": "Write a Python function that computes the nth Fibonacci number."},
        ],
        "max_new_tokens": 256,
        "do_sample": False,
    },
    timeout=120,
)
print(json.dumps(resp.json(), indent=2))

## Calling the API from your client

### `/chat` — recommended for instruct use
```python
import requests

PUBLIC_URL = "https://<your-ngrok-id>.ngrok-free.app"  # copy from cell 6

response = requests.post(
    f"{PUBLIC_URL}/chat",
    json={
        "messages": [
            {"role": "system", "content": "You are a helpful coding assistant."},
            {"role": "user",   "content": "Implement quicksort in Python with comments."},
        ],
        "max_new_tokens": 512,
        "temperature": 0.7,
        "top_p": 0.95,
        "do_sample": True,
    },
    timeout=120,
)
print(response.json()["content"])
```

### `/generate` — raw completion (no chat template)
```python
response = requests.post(
    f"{PUBLIC_URL}/generate",
    json={"prompt": "def quicksort(arr):", "max_new_tokens": 256, "do_sample": False},
    timeout=120,
)
print(response.json()["generated_text"])
```

### Request schema
| Field | Type | Default | Description |
|---|---|---|---|
| `messages` | list of `{role, content}` | required (`/chat` only) | `role`: `system` / `user` / `assistant` |
| `prompt` | string | required (`/generate` only) | Raw text prompt |
| `max_new_tokens` | int | 512 | Max tokens to generate (1–4096) |
| `temperature` | float | 0.7 | Sampling temperature (0–2) |
| `top_p` | float | 0.95 | Nucleus sampling threshold |
| `do_sample` | bool | true | Greedy if false |

> **Kaggle GPU tip:** Qwen2.5-Coder-14B needs ~28 GB VRAM in float16.  
> Use **T4 x2** (2 × 16 GB) with `device_map="auto"`, or add `load_in_4bit=True` + `BitsAndBytesConfig` for a single T4.